In [2]:
import os, json, time, argparse
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix

OUT = os.path.join(os.getcwd(), "exp6")
CKPT, PLOTS = os.path.join(OUT, "ckpt"), os.path.join(OUT, "plots")
os.makedirs(CKPT, exist_ok=True); os.makedirs(PLOTS, exist_ok=True)

HAR_CLASSES = ["WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS", "SITTING", "STANDING", "LAYING"]
VIDEO_CLASSES = ["Basketball", "Biking", "Walking", "Running", "TennisSwing"]
SEED = 42

def fit_or_load(name, build_fn, x_tr, y_tr, x_va, y_va, epochs, batch=32, force=False):
    mp, hp, jp = [os.path.join(CKPT, name + s) for s in (".keras", "_hist.json", "_meta.json")]
    if not force and all(os.path.exists(p) for p in (mp, hp, jp)):
        model = keras.models.load_model(mp)
        return model, json.load(open(hp)), json.load(open(jp))
    model = build_fn()
    model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
    t0 = time.time()
    h = model.fit(x_tr, y_tr, validation_data=(x_va, y_va), epochs=epochs,
                   batch_size=batch, verbose=2)
    meta = {"timestamp": time.strftime("%Y%m%d_%H%M%S"), "train_time": time.time() - t0,
            "params": int(model.count_params())}
    model.save(mp)
    json.dump(h.history, open(hp, "w"))
    json.dump(meta, open(jp, "w"))
    return model, h.history, meta


def eval_model(model, x_te, y_te, n_classes):
    y_prob = model.predict(x_te, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    acc = accuracy_score(y_te, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_te, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_te, y_pred, labels=list(range(n_classes)))
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1, "cm": cm}


def plot_curves(hist, title, path):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].plot(hist["loss"], label="train"); ax[0].plot(hist["val_loss"], label="val")
    ax[0].set_title(title + " loss"); ax[0].legend()
    ax[1].plot(hist["accuracy"], label="train"); ax[1].plot(hist["val_accuracy"], label="val")
    ax[1].set_title(title + " accuracy"); ax[1].legend()
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def plot_cm(cm, classes, title, path):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=45, ha="right")
    ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7)
    ax.set_title(title); fig.tight_layout(); fig.savefig(path); plt.close(fig)



def har_window(cls, T, F, rng):
    t = np.arange(T)
    dynamic = cls in (0, 1, 2)
    phase = rng.uniform(0, 2 * np.pi)
    base = rng.uniform(-0.2, 0.2)
    noise = 0.30

    if dynamic:
        freq = rng.uniform(0.06, 0.14)
        amp = rng.uniform(0.6, 1.1)
        trend = {0: rng.uniform(-0.003, 0.003),
                 1: rng.uniform(0.005, 0.012),
                 2: rng.uniform(-0.012, -0.005)}[cls]
    else:
        freq = rng.uniform(0.01, 0.04)
        amp = rng.uniform(0.03, 0.08)
        trend = 0.0
        base += {3: rng.uniform(-0.15, 0.15),
                 4: rng.uniform(0.05, 0.35),
                 5: rng.uniform(-0.35, -0.05)}[cls]

    x = np.zeros((T, F), np.float32)
    for f in range(F):
        x[:, f] = (base + amp * np.sin(2 * np.pi * freq * (1 + .04 * f) * t + phase + .3 * f)
                   + trend * t + rng.normal(0, noise, T))
    return x


def make_har(n, T=128, F=9, seed=SEED):
    rng = np.random.default_rng(seed)
    per = max(1, n // 6)
    X = [har_window(c, T, F, rng) for c in range(6) for _ in range(per)]
    y = [c for c in range(6) for _ in range(per)]
    X, y = np.array(X, np.float32), np.array(y)
    idx = rng.permutation(len(X))
    return X[idx], y[idx]


def prep_har(n, T=128):
    X, y = make_har(n, T=T)
    Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=.3, stratify=y, random_state=SEED)
    Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=.5, stratify=ytmp, random_state=SEED)
    mu, sd = Xtr.mean((0, 1), keepdims=True), Xtr.std((0, 1), keepdims=True) + 1e-8
    Xtr, Xva, Xte = (Xtr - mu) / sd, (Xva - mu) / sd, (Xte - mu) / sd
    print("Input tensor shape:")
    print(f"Training   : {Xtr.shape}\nValidation : {Xva.shape}\nTesting    : {Xte.shape}")
    return Xtr, ytr, Xva, yva, Xte, yte



def bptt_demo():
    x, h0, Wx, Wh, b = [0.5, 0.7, 0.2], 0.0, 0.5, 0.8, 0.1
    h = h0
    print("BPTT numerical exercise:")
    for i, xt in enumerate(x, 1):
        h = np.tanh(Wx * xt + Wh * h + b)
        print(f"  h{i} = {h:.6f}")



def build_classifier(rec, units=32, T=128, F=9, n_classes=6, stacked=False, bidir=False):
    Layer = {"simplernn": layers.SimpleRNN, "lstm": layers.LSTM, "gru": layers.GRU}[rec]
    wrap = (lambda l: layers.Bidirectional(l)) if bidir else (lambda l: l)
    inp = keras.Input((T, F))
    x = inp
    if stacked:
        x = wrap(Layer(units, return_sequences=True))(x)
        x = wrap(Layer(units))(x)
    else:
        x = wrap(Layer(units))(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inp, out)



def gen_video(cls, n_frames, size, rng):
    c, span = size / 2, size * 0.32
    motion = {
        0: lambda t: (c, c + span * abs(np.sin(2 * np.pi * t))),
        1: lambda t: (c + span * np.sin(4 * np.pi * t), c),
        2: lambda t: (c + span * .5 * np.sin(2 * np.pi * t), c + 10 * np.sin(4 * np.pi * t)),
        3: lambda t: (c + span * np.sin(5 * np.pi * t), c + 14 * abs(np.sin(4 * np.pi * t))),
        4: lambda t: (c + span * np.sin(np.pi * t), c - span * np.cos(np.pi * t)),
    }[cls]
    frames = np.zeros((n_frames, size, size, 3), np.uint8)
    yy, xx = np.mgrid[0:size, 0:size]
    for f in range(n_frames):
        cx, cy = motion(f / max(1, n_frames - 1))
        img = np.full((size, size, 3), 30, np.uint8)
        mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= 16 ** 2
        img[mask] = [230, 140, 20]
        frames[f] = img
    return frames


def cnn_features(frames, extractor, size):
    n, t = frames.shape[:2]
    flat = keras.applications.mobilenet_v2.preprocess_input(
        frames.reshape(-1, size, size, 3).astype(np.float32))
    feats = extractor.predict(flat, batch_size=64, verbose=0)
    return feats.reshape(n, t, -1)


def video_pipeline(per_class, n_frames, size, epochs, force):
    rng = np.random.default_rng(SEED)
    cache = os.path.join(CKPT, f"video_feats_{per_class}_{n_frames}_{size}.npz")
    if os.path.exists(cache):
        d = np.load(cache); Xf, y = d["Xf"], d["y"]
    else:
        frames = np.stack([gen_video(c, n_frames, size, rng)
                            for c in range(5) for _ in range(per_class)])
        y = np.array([c for c in range(5) for _ in range(per_class)])
        try:
            base = keras.applications.MobileNetV2(input_shape=(size, size, 3),
                                                    include_top=False, weights="imagenet", pooling="avg")
        except Exception:
            print("  ImageNet weights unavailable, using random-init MobileNetV2.")
            base = keras.applications.MobileNetV2(input_shape=(size, size, 3),
                                                    include_top=False, weights=None, pooling="avg")
        base.trainable = False
        Xf = cnn_features(frames, base, size)
        np.savez_compressed(cache, Xf=Xf, y=y)
    D = Xf.shape[-1]
    print(f"CNN feature dimension: {D}   Tensor to recurrent network: {Xf.shape}")

    # Plot 7: sample frames from one video
    example = gen_video(0, n_frames, size, rng)
    fig, axes = plt.subplots(1, min(5, n_frames), figsize=(12, 3))
    step = max(1, n_frames // len(axes))
    for i, ax in enumerate(axes):
        ax.imshow(example[min(i * step, n_frames - 1)]); ax.axis("off")
    fig.suptitle(f"Sample frames - {VIDEO_CLASSES[0]}")
    fig.savefig(os.path.join(PLOTS, "video_sample_frames.png")); plt.close(fig)

    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, test_size=.3, random_state=SEED)
    va, te = train_test_split(tmp, test_size=.5, random_state=SEED)
    results = {}
    for rec in ["lstm", "gru"]:
        name = f"video_{rec}"
        model, hist, meta = fit_or_load(name, lambda rec=rec: build_classifier(
            rec, T=n_frames, F=D, n_classes=5), Xf[tr], y[tr], Xf[va], y[va],
            epochs=epochs, batch=16, force=force)
        plot_curves(hist, f"video_{rec}", os.path.join(PLOTS, f"video_{rec}_curves.png"))
        res = eval_model(model, Xf[te], y[te], 5)
        plot_cm(res["cm"], VIDEO_CLASSES, f"CNN-{rec.upper()} confusion",
                os.path.join(PLOTS, f"video_{rec}_cm.png"))
        print(f"CNN-{rec.upper()}: acc={res['accuracy']*100:.1f}% f1={res['f1']*100:.1f}% "
              f"params={meta['params']}")

        # example prediction with confidence
        ei = te[0]
        conf = model.predict(Xf[ei:ei + 1], verbose=0)[0]
        pred_cls = int(np.argmax(conf))
        print(f"  Example -> Predicted: {VIDEO_CLASSES[pred_cls]}  "
              f"Actual: {VIDEO_CLASSES[int(y[ei])]}  Confidence: {conf[pred_cls]:.4f}")

        res["params"] = meta["params"]
        results[rec] = res
    return results



PAD, START, END = 0, 10, 11


def gen_seq2seq(n, in_len, out_len, seed=SEED):
    rng = np.random.default_rng(seed)
    enc = rng.integers(1, 10, size=(n, in_len))
    tgt = enc[:, ::-1][:, :out_len]
    tgt_full = np.concatenate([tgt, np.full((n, 1), END)], axis=1)
    dec_in = np.concatenate([np.full((n, 1), START), tgt_full[:, :-1]], axis=1)
    return enc.astype(np.int32), dec_in.astype(np.int32), tgt_full.astype(np.int32)


def build_seq2seq(in_len, out_len1, vocab=12, emb=16, units=64):
    enc_in = keras.Input((in_len,), name="enc_in")
    e = layers.Embedding(vocab, emb, name="enc_emb")(enc_in)
    _, h, c = layers.LSTM(units, return_state=True, name="enc_lstm")(e)
    dec_in = keras.Input((out_len1,), name="dec_in")
    d = layers.Embedding(vocab, emb, name="dec_emb")(dec_in)
    do, _, _ = layers.LSTM(units, return_sequences=True, return_state=True,
                            name="dec_lstm")(d, initial_state=[h, c])
    out = layers.Dense(vocab, activation="softmax", name="dec_dense")(do)
    return keras.Model([enc_in, dec_in], out)


def seq2seq_infer_models(model, units):
    enc_in = model.inputs[0]
    e = model.get_layer("enc_emb")(enc_in)
    _, h, c = model.get_layer("enc_lstm")(e)
    enc_model = keras.Model(enc_in, [h, c])
    h_in, c_in, tok_in = keras.Input((units,)), keras.Input((units,)), keras.Input((1,))
    d = model.get_layer("dec_emb")(tok_in)
    do, h2, c2 = model.get_layer("dec_lstm")(d, initial_state=[h_in, c_in])
    pred = model.get_layer("dec_dense")(do)
    dec_model = keras.Model([tok_in, h_in, c_in], [pred, h2, c2])
    return enc_model, dec_model


def greedy_decode(seq, enc_model, dec_model, out_len):
    h, c = enc_model.predict(seq[None, :], verbose=0)
    tok, out = np.array([[START]]), []
    for _ in range(out_len + 1):
        p, h, c = dec_model.predict([tok, h, c], verbose=0)
        nxt = int(np.argmax(p[0, -1]))
        if nxt == END:
            break
        out.append(nxt); tok = np.array([[nxt]])
    return out


def seq2seq_pipeline(n, in_len, out_len, units, epochs, force):
    enc, dec, tgt = gen_seq2seq(n, in_len, out_len)
    ntr, nva = int(.8 * n), int(.1 * n)
    name = f"seq2seq_{in_len}_{out_len}"
    model, hist, meta = fit_or_load(
        name, lambda: build_seq2seq(in_len, out_len + 1, units=units),
        [enc[:ntr], dec[:ntr]], tgt[:ntr],
        [enc[ntr:ntr + nva], dec[ntr:ntr + nva]], tgt[ntr:ntr + nva],
        epochs=epochs, batch=64, force=force)
    enc_model, dec_model = seq2seq_infer_models(model, units)

    print("Sample  Input               Predicted")
    for i in range(5):
        gi = ntr + nva + i
        pred = greedy_decode(enc[gi], enc_model, dec_model, out_len)
        print(f"{i+1:<8}{[int(v) for v in enc[gi]]}   {pred}")

    tok_ok = tok_n = seq_ok = seq_n = 0
    for gi in range(ntr + nva, n):
        target = [t for t in tgt[gi] if t != END]
        pred = greedy_decode(enc[gi], enc_model, dec_model, out_len)
        pred = (pred + [-1] * len(target))[:len(target)]
        tok_ok += sum(p == t for p, t in zip(pred, target)); tok_n += len(target)
        seq_ok += int(pred == target); seq_n += 1
    print(f"Token accuracy: {tok_ok/tok_n*100:.2f}%  Sequence accuracy: {seq_ok/seq_n*100:.2f}%")
    return {"token_acc": tok_ok / tok_n, "seq_acc": seq_ok / seq_n}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--quick", action="store_true")
    ap.add_argument("--force-retrain", action="store_true")
    args, _ = ap.parse_known_args()
    tf.random.set_seed(SEED); np.random.seed(SEED)
    force = args.force_retrain

    if args.quick:
        n_har, epochs, seqlen_ep = 240, 3, 2
        vid_per_class, vid_frames, vid_size, vid_ep = 6, 10, 224, 2
        seq_n, seq_len, seq_units, seq_ep = 400, 6, 32, 5
    else:
        n_har, epochs, seqlen_ep = 3000, 30, 15
        vid_per_class, vid_frames, vid_size, vid_ep = 10, 10, 224, 15
        seq_n, seq_len, seq_units, seq_ep = 5000, 6, 64, 20

    Xtr, ytr, Xva, yva, Xte, yte = prep_har(n_har)
    T, F = Xtr.shape[1], Xtr.shape[2]

    bptt_demo()

    results = {}
    for rec in ["simplernn", "lstm", "gru"]:
        model, hist, meta = fit_or_load(
            f"sensor_{rec}", lambda rec=rec: build_classifier(rec, T=T, F=F),
            Xtr, ytr, Xva, yva, epochs=epochs, force=force)
        plot_curves(hist, rec, os.path.join(PLOTS, f"{rec}_curves.png"))
        res = eval_model(model, Xte, yte, 6)
        plot_cm(res["cm"], HAR_CLASSES, rec, os.path.join(PLOTS, f"{rec}_cm.png"))
        print(f"{rec}: acc={res['accuracy']*100:.1f}% f1={res['f1']*100:.1f}% "
              f"params={meta['params']} time={meta['train_time']:.1f}s")
        results[rec] = (res, meta)

    pd.DataFrame({rec: {"accuracy": r[0]["accuracy"], "precision": r[0]["precision"],
                         "recall": r[0]["recall"], "f1": r[0]["f1"],
                         "params": r[1]["params"], "train_time": r[1]["train_time"]}
                  for rec, r in results.items()}).T.to_csv(os.path.join(OUT, "sensor_results.csv"))

    # Plot 5: performance comparison bar chart (accuracy / F1 / normalized time)
    labels = ["simplernn", "lstm", "gru"]
    acc = [results[m][0]["accuracy"] * 100 for m in labels]
    f1 = [results[m][0]["f1"] * 100 for m in labels]
    tmax = max(results[m][1]["train_time"] for m in labels) or 1.0
    ntime = [results[m][1]["train_time"] / tmax * 100 for m in labels]
    x = np.arange(3); w = 0.25
    fig, ax = plt.subplots()
    ax.bar(x - w, acc, width=w, label="Accuracy %")
    ax.bar(x, f1, width=w, label="Macro F1 %")
    ax.bar(x + w, ntime, width=w, label="Normalized train time %")
    ax.set_xticks(x); ax.set_xticklabels([m.upper() for m in labels]); ax.legend()
    fig.savefig(os.path.join(PLOTS, "model_comparison.png")); plt.close(fig)

    # Sequence length study (F1 and training time per length)
    seqlen_f1, seqlen_time = {}, {}
    for Tlen in [32, 64, 128]:
        f1_row, time_row = {}, {}
        for rec in ["simplernn", "lstm", "gru"]:
            model, hist, meta = fit_or_load(
                f"seqlen_{rec}_{Tlen}", lambda rec=rec, Tlen=Tlen: build_classifier(rec, T=Tlen, F=F),
                Xtr[:, :Tlen], ytr, Xva[:, :Tlen], yva, epochs=seqlen_ep, force=force)
            res = eval_model(model, Xte[:, :Tlen], yte, 6)
            f1_row[rec] = res["f1"] * 100
            time_row[rec] = meta["train_time"]
        seqlen_f1[Tlen] = f1_row
        seqlen_time[Tlen] = time_row
        print(f"T={Tlen}: F1={f1_row}  time(s)={time_row}")
    seqlen_f1_df = pd.DataFrame(seqlen_f1).T
    seqlen_time_df = pd.DataFrame(seqlen_time).T
    seqlen_f1_df.to_csv(os.path.join(OUT, "seqlen_f1.csv"))
    seqlen_time_df.to_csv(os.path.join(OUT, "seqlen_training_time.csv"))

    fig, ax = plt.subplots()
    for c in seqlen_f1_df.columns:
        ax.plot(seqlen_f1_df.index, seqlen_f1_df[c], marker="o", label=c)
    ax.set_xlabel("Sequence length"); ax.set_ylabel("Test F1 (%)"); ax.legend()
    fig.savefig(os.path.join(PLOTS, "seqlen_f1.png")); plt.close(fig)

    fig, ax = plt.subplots()
    for c in seqlen_time_df.columns:
        ax.plot(seqlen_time_df.index, seqlen_time_df[c], marker="o", label=c)
    ax.set_xlabel("Sequence length"); ax.set_ylabel("Training time (s)"); ax.legend()
    fig.savefig(os.path.join(PLOTS, "seqlen_training_time.png")); plt.close(fig)

    # Video CNN-RNN
    video_results = video_pipeline(vid_per_class, vid_frames, vid_size, vid_ep, force)

    # seq2seq (input length == output length)
    seq2seq_res = seq2seq_pipeline(seq_n, seq_len, seq_len, seq_units, seq_ep, force)

    # seq2seq with output length != input length
    print("\nExercise: seq2seq with output length != input length")
    seq2seq_varlen_res = seq2seq_pipeline(seq_n, seq_len, seq_len - 2, seq_units, seq_ep, force)

    pd.DataFrame([
        {"task": "reversal (in_len=out_len)", **seq2seq_res},
        {"task": f"reversal (out_len=in_len-2)", **seq2seq_varlen_res},
    ]).to_csv(os.path.join(OUT, "seq2seq_results.csv"), index=False)

    # Units 16/64, stacked, bidirectional (LSTM)
    print("\nExercise: unit variation (LSTM)")
    for u in [16, 32, 64]:
        model, hist, meta = fit_or_load(
            f"lstm_u{u}", lambda u=u: build_classifier("lstm", units=u, T=T, F=F),
            Xtr, ytr, Xva, yva, epochs=epochs, force=force)
        res = eval_model(model, Xte, yte, 6)
        print(f"  units={u}: acc={res['accuracy']*100:.1f}% f1={res['f1']*100:.1f}% params={meta['params']}")

    print("Exercise: 2-layer stacked LSTM")
    model, hist, meta = fit_or_load(
        "lstm_stacked", lambda: build_classifier("lstm", T=T, F=F, stacked=True),
        Xtr, ytr, Xva, yva, epochs=epochs, force=force)
    res = eval_model(model, Xte, yte, 6)
    print(f"  stacked: acc={res['accuracy']*100:.1f}% f1={res['f1']*100:.1f}% params={meta['params']}")

    print("Exercise: Bidirectional LSTM")
    model, hist, meta = fit_or_load(
        "lstm_bidir", lambda: build_classifier("lstm", T=T, F=F, bidir=True),
        Xtr, ytr, Xva, yva, epochs=epochs, force=force)
    res = eval_model(model, Xte, yte, 6)
    print(f"  bidirectional: acc={res['accuracy']*100:.1f}% f1={res['f1']*100:.1f}% params={meta['params']}")

    # Consolidated results table (sensor models + CNN-RNN video models)
    consolidated = {rec: {"accuracy": r[0]["accuracy"] * 100, "f1": r[0]["f1"] * 100,
                           "params": r[1]["params"]} for rec, r in results.items()}
    consolidated.update({f"CNN-{rec}": {"accuracy": r["accuracy"] * 100, "f1": r["f1"] * 100,
                                         "params": r["params"]} for rec, r in video_results.items()})
    cons_df = pd.DataFrame(consolidated).T
    cons_df.to_csv(os.path.join(OUT, "consolidated_results.csv"))
    print("\nConsolidated results (sensor + CNN-RNN):")
    print(cons_df.to_string())

    print(f"\nArtifacts saved under: {OUT}")


if __name__ == "__main__":
    main()

Input tensor shape:
Training   : (2100, 128, 9)
Validation : (450, 128, 9)
Testing    : (450, 128, 9)
BPTT numerical exercise:
  h1 = 0.336376
  h2 = 0.616352
  h3 = 0.599958
Epoch 1/30
66/66 - 7s - 103ms/step - accuracy: 0.3919 - loss: 1.5047 - val_accuracy: 0.5889 - val_loss: 1.1873
Epoch 2/30
66/66 - 1s - 12ms/step - accuracy: 0.6038 - loss: 1.1186 - val_accuracy: 0.6622 - val_loss: 0.8616
Epoch 3/30
66/66 - 1s - 10ms/step - accuracy: 0.6295 - loss: 0.9509 - val_accuracy: 0.6733 - val_loss: 0.8129
Epoch 4/30
66/66 - 1s - 11ms/step - accuracy: 0.6648 - loss: 0.8386 - val_accuracy: 0.6933 - val_loss: 0.7597
Epoch 5/30
66/66 - 1s - 10ms/step - accuracy: 0.7052 - loss: 0.7595 - val_accuracy: 0.7267 - val_loss: 0.6792
Epoch 6/30
66/66 - 1s - 10ms/step - accuracy: 0.7119 - loss: 0.7068 - val_accuracy: 0.7333 - val_loss: 0.6305
Epoch 7/30
66/66 - 1s - 11ms/step - accuracy: 0.6695 - loss: 0.8126 - val_accuracy: 0.7133 - val_loss: 0.7145
Epoch 8/30
66/66 - 1s - 10ms/step - accuracy: 0.7205 -

CNN-LSTM: acc=62.5% f1=60.0% params=168677
  Example -> Predicted: Biking  Actual: Biking  Confidence: 0.5290
Epoch 1/15
3/3 - 1s - 452ms/step - accuracy: 0.2286 - loss: 1.6745 - val_accuracy: 0.2857 - val_loss: 1.4904
Epoch 2/15
3/3 - 0s - 33ms/step - accuracy: 0.2286 - loss: 1.5147 - val_accuracy: 0.4286 - val_loss: 1.3897
Epoch 3/15
3/3 - 0s - 33ms/step - accuracy: 0.3429 - loss: 1.4787 - val_accuracy: 0.4286 - val_loss: 1.3895
Epoch 4/15
3/3 - 0s - 32ms/step - accuracy: 0.3429 - loss: 1.4505 - val_accuracy: 0.4286 - val_loss: 1.3232
Epoch 5/15
3/3 - 0s - 35ms/step - accuracy: 0.4286 - loss: 1.3930 - val_accuracy: 0.4286 - val_loss: 1.2187
Epoch 6/15
3/3 - 0s - 34ms/step - accuracy: 0.4857 - loss: 1.2680 - val_accuracy: 0.5714 - val_loss: 1.1368
Epoch 7/15
3/3 - 0s - 32ms/step - accuracy: 0.5143 - loss: 1.2725 - val_accuracy: 0.5714 - val_loss: 1.0420
Epoch 8/15
3/3 - 0s - 37ms/step - accuracy: 0.5429 - loss: 1.1276 - val_accuracy: 0.5714 - val_loss: 1.0395
Epoch 9/15
3/3 - 0s - 32m

In [3]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/exp6 /content/drive/MyDrive/exp6

Mounted at /content/drive
